# Darshan AI: GPU-Accelerated Embeddings & Bulk Document Processing

This standalone Google Colab notebook provides GPU-accelerated batch embedding and vector indexing for **Darshan AI**.

### Capabilities
- 🚀 **GPU Batch Embedding**: Process thousands of document chunks in seconds using Hugging Face `sentence-transformers` on Colab T4/V100/A100 GPUs.
- 📄 **Multi-Format Extraction**: Ingest and chunk PDFs and TXT files.
- 🔍 **FAISS Vector Indexing**: Build inner-product/cosine similarity indexes.
- 💾 **Export for Darshan AI**: Export pre-computed chunks and vectors into `darshan_ai_store.json` to upload directly into Darshan AI.

> **Note**: This notebook runs independently. Darshan AI does **not** require an active Colab session to function.

In [ ]:
# Step 1: Install High-Performance GPU Dependencies
!pip install -q sentence-transformers faiss-gpu pypdf numpy torch

In [ ]:
import os
import json
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import pypdf

# Verify GPU Hardware Acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 2: Load Embedding Model onto GPU
# Default: all-MiniLM-L6-v2 (fast & lightweight) or BAAI/bge-large-en-v1.5
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Loading {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME, device=device)
print("Model loaded successfully!")

In [ ]:
# Step 3: Chunking Function with Citation Preservation
def chunk_text(text, source_name, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    text_len = len(text)
    idx = 1
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk_content = text[start:end].strip()
        if chunk_content:
            chunks.append({
                "id": f"{source_name}-c{idx}",
                "source": source_name,
                "chunk_index": idx,
                "content": chunk_content
            })
            idx += 1
        if end >= text_len:
            break
        start = max(start + 1, end - overlap)
    return chunks

# Create sample text for demonstration
sample_doc = """
Darshan AI Architecture Overview.
The application leverages a React and Vite frontend with Supabase Authentication and a Python FastAPI backend.
For RAG pipelines, uploaded PDFs and text documents are parsed into overlapping chunks, vectorized via Sentence Transformers, and stored in FAISS or Supabase pgvector.
When a user issues a prompt, the agent searches relevant citations and attaches ground-truth source references [1].
Reverse engineering capabilities are coordinated via REA (Reverse Engineer Anything) through its documented CLI (rea doctor, rea inspect) and MCP server.
"""

demo_chunks = chunk_text(sample_doc, "darshan_ai_architecture.txt")
print(f"Created {len(demo_chunks)} chunks for processing.")

In [ ]:
# Step 4: Batch Vector Embedding on GPU
texts = [c["content"] for c in demo_chunks]
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
print(f"Embeddings matrix shape: {embeddings.shape}")

In [ ]:
# Step 5: Test Semantic Query Retrieval
query = "How does Darshan AI handle reverse engineering?"
query_vec = model.encode([query], normalize_embeddings=True)[0]

# Compute cosine similarity
scores = np.dot(embeddings, query_vec)
top_indices = np.argsort(scores)[::-1][:2]

print(f"Query: {query}\n")
for rank, idx in enumerate(top_indices, 1):
    print(f"[{rank}] Score: {scores[idx]:.4f} | Source: {demo_chunks[idx]['source']}")
    print(f"Content: {demo_chunks[idx]['content']}\n")

In [ ]:
# Step 6: Export Processed Chunks & Embeddings for Darshan AI
output_payload = []
for i, chunk in enumerate(demo_chunks):
    output_payload.append({
        "id": chunk["id"],
        "source": chunk["source"],
        "chunk_index": chunk["chunk_index"],
        "content": chunk["content"],
        "embedding": embeddings[i].tolist()
    })

with open("darshan_ai_colab_export.json", "w", encoding="utf-8") as f:
    json.dump(output_payload, f, indent=2)

print("Export saved to 'darshan_ai_colab_export.json'!")
print("You can download this file from the Colab file tree to import pre-computed vectors into Darshan AI.")